# RNNTagger2XML

If you are developing an XML corpus, you may want to insert metadata you have generated back into its `<w>` nodes. This demo notebook shows how [RNNTagger](https://www.cis.uni-muenchen.de/~schmid/tools/RNNTagger/) POS output may be added to the XML document from which its input tokens were originally sourced.

## The Usual

Once again we'll clone ECHOE (as in [the `fetch_repo.ipynb` notebook](https://github.com/langeslag/ehtc/blob/main/templates/fetch_repo.ipynb)), load the XML corpus, access its text nodes using `lxml.etree` (as in [the demo notebook by that name](https://github.com/langeslag/ehtc/blob/main/demo/lxml.etree.ipynb), only this time we are not discarding any elements), and normalize the token data ([ditto](https://github.com/langeslag/ehtc/blob/main/demo/lxml.etree.ipynb)). We'll also import the `subprocess` library so we can execute external programs:

In [1]:
import re,subprocess
from pathlib import Path
from lxml import etree
from git import Repo

We'll ascertain the ECHOE repository has been cloned so we have XML documents to work with:

In [2]:
# HTTPS clone point:
remote = 'https://github.com/ECHOEProject/echoe.git'
# Desired target folder name:
local = Path.cwd().parent / 'corpora' / 'echoe'
# Only clone if the target folder doesn't already exist:
if not(local.exists()):
    repo = Repo.clone_from(remote, local)
# Else, just update the working copy from remote:
else:
    repo = Repo(local)
    assert isinstance(repo, Repo)
    repo.remotes.origin.pull()
assert not repo.bare

In [3]:
# Create a dict with characters we want replaced:
substitutions = {
    'ę': 'æ',
    'ƿ': 'w',
    'ẏ': 'y',
    'ſ': 's',
    '': 's', # Using the glyph for descending s, instead of the unicode key point
    'v': 'u',
    'j': 'i',
    'ꝛ': 'r',
    '&': 'et',
    '\uf149': 'þ',
    '\ue337': 'þ',
    '·': '',
    ' ': '',
    '\n': '',
    '\u2028': ''
}

# Write a function carrying out the desired operations:
def normalize(token):
    # Lowercase:
    token = token.lower()
    for k,v in substitutions.items():
        # Carry out replacements:
        token = token.replace(k, v)
    return token

In [4]:
# I have yet to discover how to make elements writeable without resolving entities!
parser = etree.XMLParser(remove_blank_text=False,resolve_entities='internal')

xml_folder = Path.cwd().parent / 'corpora' / 'echoe' / 'xml'
plaintext_folder = Path.cwd().parent / 'corpora' / 'rnntagger-plaintext'
output_folder = Path.cwd().parent / 'corpora' / 'rnntagger-xml'
plaintext_folder.mkdir(exist_ok=True)
output_folder.mkdir(exist_ok=True)

def simplify(branch):
    discard = ['abbr', 'am', 'orig', 'sic']  
    # Now we define their text nodes as empty strings:
    query = ['{http://www.tei-c.org/ns/1.0}' + i for i in discard]
    for hit in branch.iter(query):
        for element in hit.iter():
            element.text = ''
            element.tail = ''
    return branch

# Limiting this demonstration to just two files:
for xml_file in sorted(xml_folder.glob('018*.xml')):
    filename = xml_file.name
    tree = etree.parse(xml_file, parser=parser)
    root = simplify(tree.getroot())
    text = root.find('.//{http://www.tei-c.org/ns/1.0}text')
    
    tokens = []
    for token in text.iter('{http://www.tei-c.org/ns/1.0}w'):
        if token.get('{http://www.w3.org/XML/1998/namespace}lang') == 'la' or token.xpath('ancestor::*[@xml:lang][1]/@xml:lang')[0] == 'la':
            token_string = normalize(etree.tostring(token, method='text', encoding='unicode')).replace('⁊', 'et').replace('⹒', 'et')
        else:
            token_string = normalize(etree.tostring(token, method='text', encoding='unicode')).replace('⁊', 'and').replace('⹒', 'and')
        # If a word element is marked as the last part of a word, add its text content to the preceding token:
        if token.get('part') == 'F':
            position = len(tokens)-1
            tokens[position] = tokens[position] + token_string
        else:
            if re.search("^\W*$", token_string):
                token_string = 'BLANK'
            tokens.append(token_string)

    plaintext_filename = filename.replace('.xml', '.txt')
    xml_outfile = Path(output_folder) / filename
    plaintext_path = str(Path(plaintext_folder / plaintext_filename))
    with open(plaintext_path, 'w') as f:
        f.write(' '.join(tokens))
    
    rnntagger = subprocess.Popen(['./cmd/rnn-tagger-old-english.sh', plaintext_path], cwd='/opt/RNNTagger', stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    rnntagger_out, rnntagger_err = rnntagger.communicate()
    output = rnntagger_out.splitlines()

    tags = []
    for line in output:
        if '\t' in line.decode():
            pos = line.decode().split('\t')[1]
            tags.append(pos)

    nodes = []
    for node in text.iter('{http://www.tei-c.org/ns/1.0}w'):
        if node.get('part') == None or node.get('part') == 'I':
            nodes.append(node)
    
    mismatches = []
    if len(nodes) == len(tags):
        print(f"Length MATCH for {filename} ({len(tags)} labels for {len(nodes)} tokens). Processing...")
        pairs = zip(nodes, tags)
        for node, label in pairs:
            node.set('pos', label)
        with open(xml_outfile, 'wb') as outfile:
            tree.write(outfile, encoding='utf-8')
    else:
        print(f"Length MISMATCH for {filename} ({len(tags)} labels for {len(nodes)} tokens), skipping.")
        mismatches.append(filename)

Length MATCH for 018.40.xml (1926 labels for 1926 tokens). Processing...
Length MATCH for 018.42.xml (1222 labels for 1222 tokens). Processing...
